# Lab 15 — LangGraph plan-and-execute bridge

Rebuild [Lab 12's planner-executor pattern](../12-plan-and-execute-from-scratch/) in LangGraph. The strong framework value case: Lab 12 implemented dynamic parallel dispatch with a manual `ThreadPoolExecutor` + `threading.Lock`; LangGraph's `Send` primitive replaces that with a graph-level abstraction that dispatches a runtime-determined number of parallel sub-graph invocations.

> ⏱ Run time: 120-150 min including reading.
> 📖 Read [`concepts/multi-agent/langgraph-multi-agent.md`](../../concepts/multi-agent/langgraph-multi-agent.md) — the `Send` section in particular — and [`concepts/multi-agent/when-frameworks-earn-complexity.md`](../../concepts/multi-agent/when-frameworks-earn-complexity.md) first.

> **The dispatcher transformation is the lesson.** Step 7 is where Lab 12's ~30 lines of manual concurrency code reduce to ~10 lines of `Send` returns.

## Step 0: Setup

Same provider-agnostic setup as Lab 14. Same `langgraph` + `langchain` packages.

In [ ]:
import json
import os
import pathlib
import re
import time
import warnings
from typing import Annotated, Any, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field, ValidationError

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")

PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Step 1: The Lab 12 baseline (reference only)

We're rebuilding [Lab 12's solution](../12-plan-and-execute-from-scratch/solution/). For context:

- **Planner**: emits a JSON `Plan` of `PlanStep`s with `depends_on` and `parallel_group`. Five planner-prompt rules.
- **`Plan.validate_graph()`**: Kahn's algorithm for cycle detection + four other checks (duplicate IDs, unknown tools, unknown deps, parallel-group integrity).
- **Executor**: runs ONE step. Anti-improvement enforced structurally (`tc.name == step.tool`).
- **Dispatcher**: `ThreadPoolExecutor(max_workers=3)` + `threading.Lock` on shared state + manual completion tracking. **The biggest piece of from-scratch concurrency code.**
- **Replanner**: `_plan_signature` dedup escalates identical-plan replans to `partial_after_cap`. `MAX_REPLANS = 2`.
- **Synthesizer**: composes the final answer from completed step results, with citation preservation for fetched pages.

This lab keeps every constraint. What changes substantially: the dispatcher.

## Step 2: PlanStep and Plan schemas

Unchanged from Lab 12. Pydantic `StrictModel(extra="forbid")` rejects LLM-invented fields. The schemas validate the planner's JSON output before we touch it.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class PlanStep(StrictModel):
    id: str = Field(description="Unique step ID, e.g. 'step_1'")
    description: str = Field(description="What this step does, self-contained.")
    tool: str = Field(description="Tool to invoke. Must be in executor registry.")
    args: dict = Field(default_factory=dict, description="Arguments to the tool.")
    depends_on: list[str] = Field(default_factory=list,
                                    description="IDs of steps whose output this step needs.")
    parallel_group: str | None = Field(
        default=None,
        description="Optional group for concurrent execution. None = run alone.",
    )


class Plan(StrictModel):
    steps: list[PlanStep] = Field(description="The ordered list of steps.")


MAX_PLAN_STEPS = 8
EXECUTOR_MAX_STEPS = 4
MAX_REPLANS = 2

print(f"Caps: MAX_PLAN_STEPS={MAX_PLAN_STEPS}, "
      f"EXECUTOR_MAX_STEPS={EXECUTOR_MAX_STEPS}, "
      f"MAX_REPLANS={MAX_REPLANS}")


## Step 3: `Plan.validate_graph()`

Unchanged from Lab 12. Kahn's algorithm for cycle detection + duplicate IDs + unknown tools + unknown dependency references + parallel-group integrity violations. All five checks in one pass; each detected error becomes structured feedback to the planner's retry loop.

We attach `validate_graph` as a method on `Plan`. (For brevity we use a standalone function in this notebook; behavior is identical.)

In [ ]:
def validate_graph(plan: Plan, available_tools: set[str]) -> list[str]:
    """Return a list of validation errors (empty list = valid plan)."""
    errors: list[str] = []
    step_ids = {s.id for s in plan.steps}
    if len(step_ids) != len(plan.steps):
        errors.append("duplicate step IDs")
    for s in plan.steps:
        if s.tool not in available_tools:
            errors.append(
                f"step {s.id} uses tool '{s.tool}' not in executor registry: "
                f"{sorted(available_tools)}"
            )
        for dep in s.depends_on:
            if dep not in step_ids:
                errors.append(f"step {s.id} depends on unknown step '{dep}'")
            if dep == s.id:
                errors.append(f"step {s.id} depends on itself")
    by_group: dict[str, list[PlanStep]] = {}
    for s in plan.steps:
        if s.parallel_group is not None:
            by_group.setdefault(s.parallel_group, []).append(s)
    for group_name, group_steps in by_group.items():
        group_ids = {s.id for s in group_steps}
        for s in group_steps:
            for dep in s.depends_on:
                if dep in group_ids:
                    errors.append(
                        f"parallel_group '{group_name}' contains step {s.id} "
                        f"which depends on group-mate {dep}"
                    )
    # Kahn's algorithm for cycle detection
    incoming = {s.id: set(s.depends_on) for s in plan.steps}
    no_incoming = [sid for sid, deps in incoming.items() if not deps]
    visited: list[str] = []
    while no_incoming:
        n = no_incoming.pop()
        visited.append(n)
        for sid, deps in incoming.items():
            if n in deps:
                deps.discard(n)
                if not deps and sid not in visited and sid not in no_incoming:
                    no_incoming.append(sid)
    if len(visited) < len(plan.steps):
        unvisited = [s.id for s in plan.steps if s.id not in visited]
        errors.append(f"cycle detected involving steps: {unvisited}")
    return errors


print("validate_graph defined.")


## Step 4: State schema with reducer

The state schema is where the framework value-add for parallel dispatch is set up. The `completed` field uses a *reducer* — a function that LangGraph calls to merge parallel updates from multiple `Send`-dispatched executors.

Without a reducer, parallel writes clobber each other and only the last one survives. With one, all parallel updates merge correctly. This is the part Lab 12 needed `threading.Lock` for.

In [ ]:
def _merge_results(existing: dict, update: dict) -> dict:
    """Reducer for the 'completed' state field.

    Merges parallel updates from Send-dispatched executors. Last-write-wins
    per step_id (in practice each step runs once, so no conflicts).
    """
    merged = dict(existing or {})
    merged.update(update or {})
    return merged


class PlanState(TypedDict):
    """State carried through the plan-and-execute graph."""
    task: str                                          # original user task
    plan: list[dict]                                   # current plan (serialized)
    completed: Annotated[dict, _merge_results]         # step_id -> result dict
    failed: list[dict]                                 # failure records
    replan_count: int                                  # bounded by MAX_REPLANS
    plan_signature_history: list[str]                  # dedup for replans
    final_answer: str                                  # synthesizer output
    status: str                                        # "ok", "partial_after_cap", "error"


print("PlanState defined with reducer on 'completed'.")


**Why the reducer matters**: in Step 7 the dispatcher will return `list[Send]` to run multiple executors in parallel. Each executor updates `completed`. Without the reducer, the parallel writes clobber each other (last-write-wins for the whole dict, not per-key). With the reducer, each update is merged into the existing dict, so all parallel results survive.

Lab 12's equivalent: `threading.Lock` around the read-modify-write of the shared `completed` dict.

## Step 5: Planner node

Same prompt as Lab 12. Same retry loop on validation failure. The planner node reads the task from state, possibly with failure context from a prior failed plan (used by the replanner).

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=MODEL, temperature=0)


# Same executor tool registry as Lab 12
EXECUTOR_TOOL_REGISTRY: dict[str, dict] = {
    "web_search": {
        "description": "Search the web. Returns up to max_results items with title, url, snippet.",
        "args_schema": {
            "query": "string, 3-8 specific words",
            "recency": "one of: any, day, week, month, year (default: any)",
            "max_results": "integer 1-10 (default: 8)",
        },
    },
    "fetch_page": {
        "description": "Fetch the full text content of a single URL.",
        "args_schema": {
            "url": "string, the URL to fetch",
            "max_chars": "integer (default: 8000)",
        },
    },
}


def _format_tool_registry(registry: dict[str, dict]) -> str:
    lines = []
    for tool_name, info in registry.items():
        lines.append(f"- {tool_name}: {info['description']}")
        for arg_name, arg_desc in info["args_schema"].items():
            lines.append(f"    args.{arg_name}: {arg_desc}")
    return "\n".join(lines)


PLANNER_SYSTEM_PROMPT_TEMPLATE = """You are a planner agent. Given a user task,
emit a Plan as a JSON object:

{{
  "steps": [
    {{
      "id": "step_1",
      "description": "What this step does, self-contained.",
      "tool": "tool_name",
      "args": {{...}},
      "depends_on": ["step_id_1", ...],
      "parallel_group": "group_name" or null
    }},
    ...
  ]
}}

EXECUTOR has access to these tools (and only these):
{tool_registry}

RULES:

1. ATOMIC STEPS. One tool call per step.
2. EXPLICIT DEPENDENCIES. List all step IDs whose output you use in depends_on.
3. WELL-FORMED PARALLEL GROUPS. Steps sharing a parallel_group must NOT depend on
   each other (directly or transitively).
4. SELF-CONTAINED DESCRIPTIONS. Executor sees only one step + its deps' outputs.
5. BOUNDED PLANS. At most {max_steps} steps.

Return ONLY JSON. No prose preamble. No markdown fences.
"""


def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    return raw


def planner_node(state: PlanState) -> dict:
    """Emit a validated Plan as a list[dict]. Stores in state['plan'].

    On validation failure, retries up to 2 more times with the error attached
    to the prompt.
    """
    system_prompt = PLANNER_SYSTEM_PROMPT_TEMPLATE.format(
        tool_registry=_format_tool_registry(EXECUTOR_TOOL_REGISTRY),
        max_steps=MAX_PLAN_STEPS,
    )

    user_prompt = f"USER TASK:\n{state['task']}"
    # If this is a replan, attach failure context
    if state.get("failed") and state.get("replan_count", 0) > 0:
        failures = state["failed"][-3:]  # most recent failures
        completed_ids = list(state.get("completed", {}).keys())
        user_prompt += (
            f"\n\nA PREVIOUS PLAN FAILED. Revise:\n"
            f"recent_failures: {json.dumps(failures, default=str)[:1500]}\n"
            f"completed_steps: {completed_ids}\n"
            f"Produce a new plan that avoids the failure(s)."
        )

    last_error = "no attempt"
    for _attempt in range(3):
        response = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt),
        ])
        raw = _strip_code_fences(
            response.content if isinstance(response.content, str) else str(response.content)
        )
        try:
            obj = json.loads(raw)
            plan = Plan.model_validate(obj)
        except (json.JSONDecodeError, ValidationError) as e:
            last_error = str(e)[:300]
            user_prompt = f"{user_prompt}\n\nPrev response invalid: {last_error}"
            continue

        errors = validate_graph(plan, set(EXECUTOR_TOOL_REGISTRY.keys()))
        if errors:
            last_error = "; ".join(errors)[:500]
            user_prompt = f"{user_prompt}\n\nGraph errors: {last_error}"
            continue
        if len(plan.steps) > MAX_PLAN_STEPS:
            last_error = f"plan has {len(plan.steps)} steps; max {MAX_PLAN_STEPS}"
            continue

        return {"plan": [s.model_dump() for s in plan.steps]}

    # All retries exhausted
    return {
        "plan": [],
        "failed": (state.get("failed") or []) + [
            {"stage": "planner", "kind": "planner_failed", "detail": last_error}
        ],
        "status": "error",
    }


print("Planner node defined.")


## Step 6: Executor sub-graph

The executor runs ONE step at a time. Anti-improvement is enforced structurally: after the LLM emits a tool call, the executor validates `tc.name == step.tool` and returns a `wrong_tool` envelope if the LLM deviates.

We reuse the same `web_search` and `fetch_page` tools from Lab 14.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
from langchain_core.tools import tool

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit", "register to read",
]


@tool
def web_search(query: str, recency: str = "any", max_results: int = 8) -> str:
    """Search the web. Returns up to max_results items with title, url, snippet."""
    if not query or not query.strip():
        return json.dumps({"status": "error", "kind": "other", "detail": "empty query"})
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except (RatelimitException, TimeoutException, DDGSException) as e:
        kind = {"RatelimitException": "rate_limit",
                "TimeoutException": "timeout"}.get(type(e).__name__, "other")
        return json.dumps({"status": "error", "kind": kind, "detail": str(e)})
    except Exception as e:
        return json.dumps({"status": "error", "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if not raw:
        return json.dumps({"status": "empty", "query": query, "detail": "no results"})
    return json.dumps({
        "status": "ok",
        "results": [{"title": (r.get("title") or "").strip(),
                     "url": (r.get("href") or "").strip(),
                     "snippet": (r.get("body") or "").strip()}
                    for r in raw if r.get("href")][:max_results],
    })


@tool
def fetch_page(url: str, max_chars: int = 8000) -> str:
    """Fetch the full text content of a URL."""
    if not url or not url.startswith(("http://", "https://")):
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": "invalid url"})
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return json.dumps({"status": "error", "url": url, "kind": "timeout",
                           "detail": "timeout"})
    except requests.RequestException as e:
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return json.dumps({"status": "error", "url": url, "kind": kind,
                           "detail": f"HTTP {resp.status_code}"})
    if 500 <= resp.status_code < 600:
        return json.dumps({"status": "error", "url": url, "kind": "http_5xx",
                           "detail": f"HTTP {resp.status_code}"})
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return json.dumps({"status": "error", "url": url, "kind": "parse",
                           "detail": f"{type(e).__name__}: {e}"})
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return json.dumps({"status": "error", "url": url, "kind": "paywall",
                           "detail": "paywall markers detected"})
    if len(text) > max_chars:
        return json.dumps({"status": "too_long", "url": url, "title": title,
                           "text": text[:max_chars], "total_chars": len(text)})
    return json.dumps({"status": "ok", "url": url, "title": title, "text": text})


EXECUTOR_TOOLS = [web_search, fetch_page]
executor_llm = llm.bind_tools(EXECUTOR_TOOLS)
print("Tools defined; executor LLM bound.")


Now define the executor node. Each `Send` invocation lands here with a payload containing the step and its dependency outputs. Anti-improvement is enforced after the LLM emits a tool call.

In [ ]:
EXECUTOR_SYSTEM_PROMPT = """You are an executor worker. You receive ONE step from
a plan plus the outputs of any steps it depends on. RUN the step's specified tool
with its specified arguments.

You MAY:
- Substitute placeholder values in args with concrete values from dependency_outputs
  (e.g. if args.url is "<first URL from step_1>", pick the actual URL).

You MUST NOT:
- Use a different tool than the one specified.
- "Improve" the args beyond filling in placeholders.

Call the specified tool ONCE and return its result.
"""


# The executor node receives a payload from each Send dispatch.
# Payload shape: {"step": dict, "deps": dict}
class ExecutorPayload(TypedDict):
    step: dict           # the PlanStep as a dict
    deps: dict           # dependency outputs (step_id -> result dict)


def executor_node(payload: ExecutorPayload) -> dict:
    """Run ONE step. Returns {'completed': {step_id: result}} or
    {'failed': [failure_record]}.

    The returned dict is merged into PlanState via the reducer on 'completed'.
    """
    step = payload["step"]
    step_id = step["id"]
    declared_tool = step["tool"]
    deps_summary = json.dumps(payload.get("deps", {}), default=str)[:4000]

    user_prompt = (
        f"STEP TO EXECUTE:\n"
        f"  id: {step_id}\n"
        f"  description: {step['description']}\n"
        f"  tool: {declared_tool}\n"
        f"  args: {json.dumps(step.get('args', {}))}\n\n"
        f"DEPENDENCY OUTPUTS:\n{deps_summary}\n\n"
        f"Call the specified tool ONCE."
    )

    response = executor_llm.invoke([
        SystemMessage(content=EXECUTOR_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    if not response.tool_calls:
        return {"failed": [{
            "step_id": step_id, "kind": "no_tool_call",
            "detail": "executor returned no tool call",
        }]}

    tc = response.tool_calls[0]
    if tc["name"] != declared_tool:
        return {"failed": [{
            "step_id": step_id, "kind": "wrong_tool",
            "detail": f"executor used {tc['name']} but step.tool was {declared_tool}",
        }]}

    # Execute the tool — find it in the registry
    tool_fn = next((t for t in EXECUTOR_TOOLS if t.name == declared_tool), None)
    if tool_fn is None:
        return {"failed": [{
            "step_id": step_id, "kind": "unknown_tool",
            "detail": f"tool {declared_tool} not in EXECUTOR_TOOLS registry",
        }]}

    try:
        result_str = tool_fn.invoke(tc["args"])
        result = json.loads(result_str) if isinstance(result_str, str) else result_str
    except Exception as e:
        return {"failed": [{
            "step_id": step_id, "kind": "tool_exception",
            "detail": f"{type(e).__name__}: {e}",
        }]}

    return {"completed": {step_id: {
        "tool": declared_tool, "args": tc["args"], "result": result,
    }}}


print("Executor node defined.")


## Step 7: Dispatcher node — the framework value-add

This is the heart of the lab. Lab 12's dispatcher was ~70 lines of `ThreadPoolExecutor` + `threading.Lock` + manual ready-step computation + completion tracking. LangGraph's `Send` reduces it to ~10 lines.

The dispatcher node returns `list[Send]` — one `Send` per ready step. LangGraph launches each `Send` as a parallel sub-graph invocation, and the results merge back into `PlanState['completed']` via the reducer we defined in Step 4.

In [ ]:
from langgraph.types import Send


def _ready_steps(plan: list[dict], completed: dict, failed_ids: set[str]) -> list[dict]:
    """Compute steps whose dependencies are satisfied and that haven't run yet."""
    return [
        s for s in plan
        if (s["id"] not in completed and s["id"] not in failed_ids
            and all(dep in completed for dep in s.get("depends_on", [])))
    ]


def dispatcher_node(state: PlanState) -> list[Send]:
    """Compute ready steps and return one Send per ready step.

    LangGraph dispatches each Send as a parallel sub-graph invocation;
    results merge back into state['completed'] via the reducer.
    """
    plan = state["plan"]
    completed = state.get("completed", {})
    failed_ids = {f["step_id"] for f in (state.get("failed") or []) if "step_id" in f}

    ready = _ready_steps(plan, completed, failed_ids)
    if not ready:
        # Nothing ready — control flows to the next edge (the replanner)
        return []

    return [
        Send("executor", {
            "step": s,
            "deps": {dep: completed.get(dep) for dep in s.get("depends_on", [])},
        })
        for s in ready
    ]


print("Dispatcher node defined.")
print()
print("Lab 12's equivalent was ~70 lines of ThreadPoolExecutor + threading.Lock.")
print("This is ~10 lines. The reducer on PlanState['completed'] handles merging.")


**The trade-off explicit**: the bounded-concurrency cap (`MAX_PARALLEL_EXECUTORS = 3` in Lab 12) is no longer visible at the dispatch site. If a single superstep has 5 ready steps, LangGraph dispatches all 5 concurrently. For most workloads this is fine — the LLM provider's rate limits are the real bound — but if you need to enforce a hard concurrency cap, you batch the `Send` returns yourself by returning at most N `Send` objects per superstep and letting subsequent supersteps drain the rest.

## Step 8: Replanner with plan-signature dedup

Lab 12's replanner was a Python `while` loop with `_plan_signature` dedup (hashed structural plan content to detect identical replans). LangGraph's replanner is a node that returns `Command(goto=...)`:

- All steps completed → `Command(goto="synthesize")`
- Failures present + replans remaining + new plan signature → `Command(goto="planner", update={...})` and increment `replan_count`
- Cap fired (or identical plan signature seen) → `Command(goto="synthesize", update={"status": "partial_after_cap"})`

In [ ]:
import hashlib

from langgraph.types import Command


def _plan_signature(plan: list[dict]) -> str:
    """Stable hash of structural plan content. Excludes descriptions."""
    structural = [
        {"id": s["id"], "tool": s["tool"], "args": s["args"],
         "depends_on": sorted(s.get("depends_on", [])),
         "parallel_group": s.get("parallel_group")}
        for s in plan
    ]
    return hashlib.sha256(
        json.dumps(structural, sort_keys=True).encode()
    ).hexdigest()[:16]


def replanner_node(
    state: PlanState,
) -> Command[Literal["planner", "synthesize"]]:
    """Decide between (a) all done → synthesize, (b) failures present → replan,
    (c) cap fired or duplicate plan → partial_after_cap.
    """
    plan = state["plan"]
    completed = state.get("completed", {})
    failed = state.get("failed") or []
    replan_count = state.get("replan_count", 0)
    signature_history = state.get("plan_signature_history") or []

    current_sig = _plan_signature(plan)
    new_history = signature_history + [current_sig]
    failed_ids = {f["step_id"] for f in failed if "step_id" in f}
    completed_or_failed = set(completed.keys()) | failed_ids
    all_steps_handled = all(s["id"] in completed_or_failed for s in plan)

    # Case (a): every step is done — proceed to synthesize
    if all_steps_handled and not failed:
        return Command(goto="synthesize", update={"plan_signature_history": new_history})

    # Case (c.1): cap fired
    if replan_count >= MAX_REPLANS:
        return Command(
            goto="synthesize",
            update={
                "status": "partial_after_cap",
                "plan_signature_history": new_history,
            },
        )

    # Case (c.2): identical plan signature — replanner is thrashing
    if current_sig in signature_history:
        return Command(
            goto="synthesize",
            update={
                "status": "partial_after_cap",
                "plan_signature_history": new_history,
            },
        )

    # Case (b): failures present, replans remaining — go back to planner
    if failed:
        return Command(
            goto="planner",
            update={
                "replan_count": replan_count + 1,
                "plan_signature_history": new_history,
            },
        )

    # Defensive fallback: shouldn't reach here normally
    return Command(goto="synthesize", update={"plan_signature_history": new_history})


print("Replanner node defined.")


## Step 9: Synthesizer

Same prompt as Lab 12. Composes the final answer from completed step results. Citation preservation for fetched pages — the synthesizer's job is the same regardless of framework.

In [ ]:
SYNTHESIZER_SYSTEM_PROMPT = """You are a synthesizer. You receive the original
task plus a dict of executed step results from a plan. Produce a clear answer
using only information from the step results.

Rules:
1. Cite fetched pages inline as [1], [2]; list at end as [N] Title — URL.
2. Do not invent claims not supported by step results.
3. If a step failed, say so. Do not paper over gaps.
"""


def synthesize_node(state: PlanState) -> dict:
    """Compose the final answer from completed step results."""
    task = state["task"]
    completed = state.get("completed", {})
    failed = state.get("failed") or []

    step_summary_lines = []
    citations: list[dict] = []
    for step_id, step_result in completed.items():
        tool_name = step_result.get("tool")
        result = step_result.get("result", {})
        result_str = json.dumps(result, default=str)[:3000]
        step_summary_lines.append(f"--- {step_id} ({tool_name}) ---\n{result_str}")
        if tool_name == "fetch_page" and isinstance(result, dict) and result.get("status") in ("ok", "too_long"):
            citations.append({"url": result.get("url", ""),
                              "title": result.get("title", "")})

    step_summary = "\n\n".join(step_summary_lines) or "(no completed steps)"
    failure_block = ""
    if failed:
        failure_block = "\n\nFAILED STEPS:\n" + "\n".join(
            f"- {f.get('step_id', '?')}: {f.get('kind')} "
            f"({(f.get('detail') or '')[:200]})"
            for f in failed
        )

    user_prompt = (
        f"USER TASK:\n{task}\n\n"
        f"COMPLETED STEP RESULTS:\n{step_summary}{failure_block}\n\n"
        f"Compose the final answer. Cite fetched URLs inline."
    )
    response = llm.invoke([
        SystemMessage(content=SYNTHESIZER_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])
    answer = (response.content
              if isinstance(response.content, str)
              else str(response.content)).strip()
    return {
        "final_answer": answer,
        "status": state.get("status") or "ok",
    }


print("Synthesizer node defined.")


## Step 10: Wire the graph

The shape: `planner → dispatcher → executor (parallel) → replanner → planner | synthesize`. The `dispatcher → executor` edge is the `Send` dispatch — LangGraph fans out across `Send` payloads and joins them via the reducer on `completed`. The `replanner → planner` edge is the replan path; `replanner → synthesize` is the terminal path.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


def build_plan_execute_graph(checkpointer=None):
    """Build and compile the plan-and-execute graph."""
    builder = StateGraph(PlanState)

    builder.add_node("planner", planner_node)
    builder.add_node("executor", executor_node)
    builder.add_node("replanner", replanner_node)
    builder.add_node("synthesize", synthesize_node)

    builder.add_edge(START, "planner")
    # planner → dispatcher (conditional, returns list[Send])
    builder.add_conditional_edges("planner", dispatcher_node, ["executor"])
    # executor → replanner (after all parallel Sends complete)
    builder.add_edge("executor", "replanner")
    # replanner uses Command to route to either "planner" or "synthesize"
    builder.add_edge("synthesize", END)

    return builder.compile(checkpointer=checkpointer)


graph = build_plan_execute_graph(checkpointer=InMemorySaver())
print("Graph compiled with checkpointer.")
print(f"Nodes: {list(graph.nodes.keys())}")


**A subtlety worth noting**: `dispatcher_node` is registered as a *conditional edge* function via `add_conditional_edges`. This is the LangGraph idiom for "a function that returns Send objects to dispatch in parallel." Returning `list[Send]` from such a function triggers parallel execution; returning a single string is the simpler conditional-edge case (route to one named node).

## Step 11: Run end-to-end

Same task as Lab 12 — compound query that triggers parallel dispatch.

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP). "
    "Summarize the top 3 key points with citations."
)

config_thread = {"configurable": {"thread_id": "demo-thread-1"}, "recursion_limit": 40}

start = time.time()
result = graph.invoke({"task": task}, config=config_thread)
elapsed = time.time() - start

print(f"Status: {result.get('status', 'ok')}")
print(f"Replans: {result.get('replan_count', 0)}")
print(f"Plan steps: {len(result.get('plan', []))}")
print(f"Completed: {len(result.get('completed', {}))}")
print(f"Failed: {len(result.get('failed', []))}")
print(f"Wall-clock: {elapsed:.1f}s")
print("=" * 70)
print(result.get("final_answer", "[no final answer]"))


**Sample output (LLM responses will vary; trajectory should be stable):**

```
Status: ok
Replans: 0
Plan steps: 4
Completed: 4
Failed: 0
Wall-clock: 7.2s
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by
Anthropic in late 2024. Three recent developments stand out:

1. Expanded server ecosystem...
2. Production deployments...
3. Tool authentication patterns...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Specification — https://modelcontextprotocol.io/...
[3] ...
```

Wall-clock around 7 seconds for a 4-step plan with 3 parallel fetches — same as Lab 12's wall-clock for an equivalent plan, because the LLM and network calls are the same. The difference is in code complexity, not runtime.

## Step 12: Line-by-line comparison

| Component | Lab 12 (from-scratch) | Lab 15 (LangGraph) | Net change |
|---|---|---|---|
| `Plan`/`PlanStep` schemas | ~30 lines (Pydantic) | ~30 lines (unchanged) | No change |
| `validate_graph` | ~50 lines (Kahn + 4 checks) | ~50 lines (unchanged) | No change |
| Planner prompt | ~50 lines (five rules) | ~50 lines (unchanged) | No change |
| Planner retry loop | ~25 lines | ~25 lines (similar shape) | No change |
| Executor | ~40 lines | ~40 lines (sub-graph node) | No change |
| **Dispatcher** | **~70 lines** (`ThreadPoolExecutor`, `threading.Lock`, ready-step computation) | **~10 lines** (`return [Send(...) for s in ready]`) | **Framework wins ~60 lines** |
| Replanner | ~25 lines (Python loop + plan signature) | ~30 lines (`Command(goto=...)` node) | No change (shape, not size) |
| Synthesizer | ~25 lines | ~25 lines (unchanged) | No change |
| Plan-state persistence | Not implemented | ~5 lines (`InMemorySaver`) | Framework adds capability |

The dispatcher transformation is the dominant change. ~70 lines of manual concurrency code reduces to ~10 lines of `Send` returns. Everything else carries over with the same shape.

**What you should take away:**

1. **The planner's prompt did not shrink.** Same five rules, same retry logic. Prompt engineering remains the dominant cost.
2. **The `validate_graph` function did not change.** Kahn's algorithm + four checks. Pure-Python validation transfers verbatim.
3. **The dispatcher transformation is where the framework earns its complexity.** This is the case worth migrating for. If your multi-agent workload has runtime-determined parallel dispatch, `Send` is a significant simplification.
4. **The reducer on `completed` is doing the work that `threading.Lock` did in Lab 12.** The merge semantics are the same; the framework just provides the merge primitive natively.
5. **The replanner moves from a Python loop to a graph edge.** Same logic; different shape. Cleaner topologically; debugging is slightly harder.

## What you just built

A LangGraph plan-and-execute system with the same capability as Lab 12's from-scratch version. The planner, validator, executor, and synthesizer carry over with essentially unchanged code. The dispatcher — Lab 12's most complex concurrency code — reduces to ~10 lines of `Send` returns plus a reducer on the state field.

This is the strong framework value case for multi-agent. `Send` is a primitive that doesn't have a clean from-scratch analog; rebuilding it from scratch means rebuilding a chunk of LangGraph itself.

## Production readiness — out of scope here

For a real deployment you'd also want: a production checkpointer; LangSmith tracing for the parallel-dispatch trajectory (in-process logs alone aren't great for understanding which `Send` did what); per-step budget enforcement via a custom node that wraps the executor; circuit breakers on tools that consistently fail; eval harnesses scoring plan quality, replan rate, and citation preservation. None of these change the canonical pattern this lab demonstrates.

## Anti-scope (handled in other places)

- **Distributed `Send` dispatch.** `Send` runs in-process; for cross-machine parallelism you'd reach for LangGraph Cloud or a custom worker pool. Out of scope.
- **Lab 11 (critic) in this dispatcher.** Adding a critic between executor and synthesizer is straightforward — it's another node — but framework-agnostic; mentioned in the concept page as an extension exercise.
- **Lab 13 (multi-agent RAG) framework rewrite.** Composes this lab + retrieval-pipeline-as-node; mentioned in the concept page as future work.

## Next

This concludes Path 03's framework-bridge module (Module 5). You've now:

- Built Lab 10's supervisor-worker from scratch.
- Built Lab 11's generator-critic from scratch.
- Built Lab 12's plan-and-execute from scratch.
- Built Lab 13's multi-agent RAG from scratch.
- Rebuilt Lab 10 in LangGraph (Lab 14) and seen the limited but useful framework value-add.
- Rebuilt Lab 12 in LangGraph (Lab 15) and seen the strong framework value-add via `Send`.

The pedagogical payoff: you can now evaluate framework adoption decisions concretely. You have working code for both sides of the comparison; the trade-offs are no longer abstract.

After completing the lab, take the [framework bridge quiz](../../quizzes/multi-agent/framework-bridge.md).

A future Module 6 will extend Lab 09's evaluation harness for multi-agent: trajectory-level metrics, plan-quality scores, replan rate, citation preservation rate. A future solutions batch will provide reference implementations for Labs 14 and 15.